#1. Basic Tasks 

##1. Load the messy e-commerce dataset and identify columns containing nulls and duplicate rows using .filter(), .distinct(), and .dropDuplicates(). 

In [0]:
import pyspark.sql.functions as F
df=spark.read.csv('/Volumes/dev/demo/raw/sales.csv',inferSchema=True,header=True)
df.display()
cols_with_nulls = [x for x in df.columns if df.filter(F.col(x).isNull()).count() > 0]
print(f"\ncolumns with atleast one value is null:{cols_with_nulls}")
# Identify duplicate rows
total_rows = df.count()
distinct_rows = df.distinct().count()
duplicate_count = total_rows - distinct_rows
print(f"\nTotal rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print("\nSample of duplicate records:")
    df.groupBy(df.columns).count().filter(F.col("count") > 1).display()

# Drop duplicates
df_cleaned = df.dropDuplicates()
print(f"\nRows after removing duplicates: {df_cleaned.count()}")

##2. Rename at least 3 columns to a consistent naming convention (e.g., snake_case) using withColumnRenamed. 

In [0]:
df_renamed = df_cleaned \
    .withColumnRenamed("transaction_id", "transaction_code") \
    .withColumnRenamed("customer_id", "customer_code") \
    .withColumnRenamed("product_id", "product_code")

print("Original columns:", df_cleaned.columns)
print("Renamed columns:", df_renamed.columns)
df_renamed.display()

##3. Connect a Databricks Repo to a Git provider and make your first commit of a cleaning notebook. 


### Git Commands Used:
```bash
git add "Day 5 Assignment: PySpark DataFrame Transformations & Git Integration.ipynb"

# Commit with descriptive message
git commit -m "Add data cleaning notebook with PySpark transformations"

# Push to remote 
git push origin main
```

##4. Build a cleaning pipeline: drop fully-null rows, fill remaining nulls with sensible defaults, remove exact duplicates, and sort by order_date. 

In [0]:
df_sales = spark.read.csv('/Volumes/dev/demo/raw/sales.csv', inferSchema=True, header=True)
df_sales = df_sales.dropna()

df_sales = df_sales.fillna({
    'quantity': 0,
    'discount_amount': 0.0,
    'total_amount': 0.0
})
df_sales = df_sales.dropDuplicates()
df_sales = df_sales.orderBy('order_date')

df_sales.display()

##5. Perform an aggregation (revenue by category or region) and a join against a second small reference table (e.g., customers or regions). 

In [0]:
from pyspark.sql import functions as f
df_cust=spark.read.json('/Volumes/cyntexa_dev/sales/external_data/orders.json',multiLine=True)
df_cust=df_cust.select(f.col("address.city").alias("city"), f.col("address.state").alias("state"),f.col("customer_id"),f.col("name"), f.explode('orders').alias("orders"))
df_cust.display()

In [0]:
df_cust_sales=df_cust.join(df_sales,'customer_id')

df_revenue_by_region = df_cust_sales.groupBy("state").agg(
    f.round(f.sum("total_amount"),2).alias("total_revenue"),
    f.count("transaction_id").alias("transaction_count")
).orderBy(f.desc("total_revenue"))

print("\nRevenue by Region (State):")
df_revenue_by_region.display()

##6. Create a feature branch in your Databricks Repo, change the cleaning logic, and open a pull request describing what changed and why. 

In [0]:
df_sales = spark.read.csv('/Volumes/dev/demo/raw/sales.csv', inferSchema=True, header=True)

#Drop rows where ALL values are null
df_sales = df_sales.dropna(how='all')

df_sales = df_sales.fillna({
    'quantity': 0,
    'discount_amount': 0.0,
    'total_amount': 0.0
})
df_sales = df_sales.dropDuplicates()
df_sales = df_sales.orderBy('order_date')

print("\nCleaned dataset:")
df_sales.display()

## Feature Branch: Improved Cleaning Logic
### Branch: `feature/feature1`
### Changes Made:
#### **Before**
```
df_sales = df_sales.dropna()  
df_sales = df_sales.fillna({...})
```

#### **After**
```
# 1. Drop only fully-null rows 
df_sales = df_sales.dropna(how='all')

df_sales = df_sales.fillna({
    'quantity': 0,
    'discount_amount': 0.0,
    'total_amount': 0.0
})
```



#3. Advanced Tasks

##7. Extend the pipeline to handle a 'dirty data' scenario not covered in class (e.g., mixed date formats, currency symbols in numeric columns) and document your approach. 

In [0]:
from pyspark.sql import functions as F
df=spark.read.csv('/Volumes/cyntexa_dev/sales/external_data/sales_dirty.csv',inferSchema=True,header=True)
df.display()



df = df.withColumn("total_amount",F.trim(F.regexp_replace(F.col("total_amount"),r"[$₹€£\s]|USD","")))
df = df.withColumn("total_amount",F.when(F.col("total_amount").rlike(r"^\d{1,3}(,\d{3})+\.\d+$"),
        F.regexp_replace(F.col("total_amount"), ",", "")).otherwise(F.regexp_replace(F.col("total_amount"), ",", "."))
)
df = df.withColumn("total_amount",F.col("total_amount").cast("decimal(10,2)"))

df = df.withColumn("discount_amount",F.regexp_replace(F.col("discount_amount"),r"[$₹€£\s]|USD",""))
df = df.withColumn("discount_amount",F.when(F.col("discount_amount").rlike(r"^\d{1,3}(,\d{3})+\.\d+$"),
        F.regexp_replace(F.col("discount_amount"), ",", "")).otherwise(
        F.regexp_replace(F.col("discount_amount"), ",", ".")))
df = df.withColumn("discount_amount",F.col("discount_amount").cast("decimal(10,2)"))

df.display()
# Clean mixed dates - try multiple formats, return NULL if none match
df = df.withColumn(
    "order_date",
    F.coalesce(
        F.try_to_date("order_date", "yyyy-MM-dd"),
        F.try_to_date("order_date", "dd/MM/yyyy"),
        F.try_to_date("order_date", "MM-dd-yyyy"),
        F.try_to_date("order_date", "dd-MMM-yyyy"),
        F.try_to_date("order_date", "yyyy/MM/dd"),
        F.try_to_date("order_date", "dd.MM.yyyy"),
        F.try_to_date("order_date", "MMMM dd, yyyy")
    )
)


df = df.dropDuplicates(["order_id"])
df.display()


##8. Set up a branching strategy (dev/main) for the Cyntexa analytics repo and write a short guide for teammates on the pull-request review workflow before merging into main. 

### Main Branches

1. **`main`** - Production-ready code
   - Always stable and deployable
   - Protected branch (requires PR approval)
   - Represents code running in production

2. **`dev`** - Development integration branch
   - Integration branch for ongoing development
   - Should be stable enough for testing
   - Feature branches merge here first

### Feature Branches

- Branch naming convention: `feature/<descriptive-name>`
- Examples: `feature/data-cleaning`, `feature/customer-segmentation`
- Always branch FROM: `dev`
- Always merge TO: `dev` (never directly to `main`)

## Quick Reference Commands

```bash
# Create feature branch from dev
git checkout dev && git pull origin dev
git checkout -b feature/my-feature

# Commit and push changes
git add .
git commit -m "Description of changes"
git push origin feature/my-feature
```



##9. (Data Analyst) Using the cleaned dataset, produce a summary report answering 3 business questions (e.g., top-selling category per region, month-over-month growth, average order value trend) and note any data-quality caveats a stakeholder should know about.

In [0]:
#top-selling category per region
from pyspark.sql import functions as f
from pyspark.sql.window import Window

df = spark.read.table('dev.gold.sales')

df_revenue = df.groupBy('region', 'product_category').agg(
    f.round(f.sum('total_revenue'), 2).alias('total_revenue')
)

w = Window.partitionBy('region').orderBy(f.desc('total_revenue'))
df_ranked = df_revenue.withColumn('rank', f.rank().over(w))


df_top_category = df_ranked.filter(f.col('rank') == 1).select('region', 'product_category', 'total_revenue')

df_top_category.display()


In [0]:
#month-over-month growth
from pyspark.sql import functions as f
from pyspark.sql.window import Window

df = spark.read.table('dev.gold.sales')

df_agg = df.groupBy('month').agg(
    f.round(f.sum('total_revenue'), 2).alias('total_revenue')
)

# Calculate month-over-month growth
w = Window.orderBy('month')
df_growth = df_agg.withColumn('prev_month_revenue', f.lag('total_revenue').over(w))
df_growth = df_growth.withColumn('growth', 
    f.round(((f.col('total_revenue') - f.col('prev_month_revenue'))*100)/f.col('prev_month_revenue'), 2)
).withColumn("year",f.year("month")).withColumn("month_name",f.date_format("month", "MMMM"))
df_growth=df_growth.select("year","month_name","total_revenue","prev_month_revenue","growth")
df_growth.orderBy('month').display()

In [0]:

from pyspark.sql import functions as f
from pyspark.sql.window import Window

df = spark.read.table('dev.gold.sales')

df_aov = df.groupBy('month').agg(
    f.round(f.avg('total_revenue'), 2).alias('avg_order_value')
).orderBy('month')

w = Window.orderBy('month')
df_aov_trend = df_aov.withColumn('prev_month_aov', f.lag('avg_order_value').over(w))
df_aov_trend = df_aov_trend.withColumn('aov_change', 
    f.round(f.col('avg_order_value') - f.col('prev_month_aov'), 2)
).withColumn('year', f.year('month')).withColumn('month_name', f.date_format('month', 'MMMM'))

df_aov_trend = df_aov_trend.select(
    'year', 'month_name', 'avg_order_value',  
    'prev_month_aov', 'aov_change'
)

df_aov_trend.display()